<h1><center>SpaGCN Tutorial Easy Mode</center></h1>


<center>Author: Jian Hu,*, Xiangjie Li, Kyle Coleman, Amelia Schroeder, Nan Ma, David J. Irwin, Edward B. Lee, Russell T. Shinohara, Mingyao Li*

### Outline
1. Installation
2. Import modules
3. Read in data
4. Spatial domain detection using SpaGCN
5. Identify SVGs

### 1. Installation
To install SpaGCN package you must make sure that your python version is over 3.5.=. If you don’t know the version of python you can check it by:

In [1]:
import platform
platform.python_version()

'3.8.8'

Note: Because SpaGCN pends on pytorch, you should make sure torch is correctly installed.
<br>
Now you can install the current release of SpaGCN by the following three ways:
#### 1.1 PyPI: Directly install the package from PyPI.

In [ ]:
pip3 install SpaGCN
#Note: you need to make sure that the pip is for python3，or we should install SpaGCN by
python3 -m pip install SpaGCN
pip3 install SpaGCN
#If you do not have permission (when you get a permission denied error), you should install SpaGCN by
pip3 install --user SpaGCN

#### 1.2 Github
Download the package from Github and install it locally:

In [2]:
git clone https://github.com/jianhuupenn/SpaGCN
cd SpaGCN/SpaGCN_package/
python3 setup.py install --user

#### 1.3 Anaconda
If you do not have Python3.5 or Python3.6 installed, consider installing Anaconda (see Installing Anaconda). After installing Anaconda, you can create a new environment, for example, SpaGCN (you can change to any name you like).

In [ ]:
#create an environment called SpaGCN
conda create -n SpaGCN python=3.7.9
#activate your environment 
conda activate SpaGCN
git clone https://github.com/jianhuupenn/SpaGCN
cd SpaGCN/SpaGCN_package/
python3 setup.py build
python3 setup.py install
conda deactivate

### 2. Import python modules

### Mouse Brain

In [ ]:

import os,csv,re
import pandas as pd
import numpy as np
import scanpy as sc
import math
import SpaGCN as spg
from scipy.sparse import issparse
import random, torch
import warnings
warnings.filterwarnings("ignore")
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import SpaGCN as spg
import cv2
#Read in gene expression and spatial location
adata=sc.read("results/enhanced_exp_MB.h5ad")
#Read in hitology image
img=cv2.imread("data/V1_Mouse_Brain_Sagittal_Anterior_image.tif")

In [ ]:

random_indices = np.random.choice(adata.shape[0], size=5000, replace=False)
adata=adata[random_indices,:]
#Set coordinates
adata.obs["x_array"]=adata.obs["x"]
adata.obs["y_array"]=adata.obs["y"]
adata.obs["x_pixel"]=adata.obs["x"]
adata.obs["y_pixel"]=adata.obs["y"]
x_array=adata.obs["x_array"].tolist()
y_array=adata.obs["y_array"].tolist()
x_pixel=adata.obs["x_pixel"].tolist()
y_pixel=adata.obs["y_pixel"].tolist()
#Run SpaGCN
adata.obs["pred"]= spg.detect_spatial_domains_ez_mode(adata, img, x_array, y_array, x_pixel, y_pixel, n_clusters=15, histology=False, s=1, b=49, p=0.5, r_seed=100, t_seed=100, n_seed=100)
adata.obs["pred"]=adata.obs["pred"].astype('category')
#Refine domains (optional)
#shape="hexagon" for Visium data, "square" for ST data.
adata.obs["refined_pred"]=spg.spatial_domains_refinement_ez_mode(sample_id=adata.obs.index.tolist(), pred=adata.obs["pred"].tolist(), x_array=x_array, y_array=y_array, shape="hexagon")
adata.obs["refined_pred"]=adata.obs["refined_pred"].astype('category')
plot_color=["#F56867","#FEB915","#C798EE","#59BE86","#7495D3","#D1D1D1","#6D1A9C","#15821E","#3A84E6","#997273","#787878","#DB4C6C","#9E7A7A","#554236","#AF5F3C","#93796C","#F9BD3F","#DAB370","#877F6C","#268785"]
ax=spg.plot_spatial_domains_ez_mode(adata, domain_name="pred", x_name="y_pixel", y_name="x_pixel", plot_color=plot_color,size=150000/adata.shape[0], show=False, save=True,save_dir="./sample_results/pred1.png")
ax=spg.plot_spatial_domains_ez_mode(adata, domain_name="refined_pred", x_name="y_pixel", y_name="x_pixel", plot_color=plot_color,size=150000/adata.shape[0], show=False, save=True,save_dir="./sample_results/refined_pred1.png")


Calculateing adj matrix using xy only...
Run 1: l [0.01, 1000], p [0.0, 496.662160704795]
Run 2: l [0.01, 500.005], p [0.0, 141.50662231445312]
Run 3: l [0.01, 250.0075], p [0.0, 37.52476119995117]
Run 4: l [0.01, 125.00874999999999], p [0.0, 9.526144027709961]
Run 5: l [0.01, 62.509375], p [0.0, 2.2602620124816895]
Run 6: l [31.2596875, 62.509375], p [0.3790069818496704, 2.2602620124816895]
Run 7: l [31.2596875, 46.884531249999995], p [0.3790069818496704, 1.1696245670318604]
Run 8: l [31.2596875, 39.072109375], p [0.3790069818496704, 0.7355605363845825]
Run 9: l [31.2596875, 35.1658984375], p [0.3790069818496704, 0.5471858978271484]
Run 10: l [33.212792968749994, 35.1658984375], p [0.4604637622833252, 0.5471858978271484]
recommended l =  34.189345703125
Start at res =  0.7 step =  0.1
Initializing cluster centers with louvain, resolution =  0.7
Epoch  0
Epoch  10
Res =  0.7 Num of clusters =  18
Initializing cluster centers with louvain, resolution =  0.6
Epoch  0
Epoch  10
Res =  0.6

In [ ]:

raw=sc.read("results/enhanced_exp_MB.h5ad")

raw=raw[random_indices,:]
# raw.var_names_make_unique()
raw.obs["pred"]=adata.obs["pred"].astype('category')
raw.obs["x_array"]=raw.obs["x"]
raw.obs["y_array"]=raw.obs["y"]
raw.obs["x_pixel"]=raw.obs["x"]
raw.obs["y_pixel"]=raw.obs["y"]
raw.X=(raw.X.A if issparse(raw.X) else raw.X)
raw.raw=raw
sc.pp.log1p(raw)
#Set filtering criterials


In [7]:
raw.obs["pred"].unique()

[7, 9, 5, 1, 11, ..., 0, 6, 10, 8, 14]
Length: 15
Categories (15, int64): [0, 1, 2, 3, ..., 11, 12, 13, 14]

In [23]:
min_in_group_fraction=0.8
min_in_out_group_ratio=1
min_fold_change=1.3
raw.X=np.array(raw.X,dtype='float32')
all_filtered = []
for i in range(15):
    filtered_info=spg.detect_SVGs_ez_mode(raw, target=i, x_name="x_array", y_name="y_array", domain_name="pred", min_in_group_fraction=min_in_group_fraction, min_in_out_group_ratio=min_in_out_group_ratio, min_fold_change=min_fold_change)
    all_filtered.append(filtered_info)



Calculateing adj matrix using xy only...
Calculateing adj matrix using xy only...
Calculateing adj matrix using xy only...
Run 1: radius [111.80339813232422, 1365.6500244140625], num_nbr [4.056939501779359, 397.56049822064057]
Calculateing adj matrix using xy only...
Run 2: radius [111.80339813232422, 738.7267112731934], num_nbr [4.056939501779359, 140.83629893238435]
Calculateing adj matrix using xy only...
Run 3: radius [111.80339813232422, 425.2650547027588], num_nbr [4.056939501779359, 52.629893238434164]
Calculateing adj matrix using xy only...
Run 4: radius [111.80339813232422, 268.5342264175415], num_nbr [4.056939501779359, 22.033807829181494]
Calculateing adj matrix using xy only...
recommended radius =  190.16881227493286 num_nbr=11.809608540925266
radius= 190.16881227493286 average number of neighbors for each spot is 11.809608540925266
 Cluster 0 has neighbors:
Dmain  5 :  362
Dmain  12 :  232
Dmain  14 :  104
SVGs for domain  0 : ['Vip', 'Cux2']
Calculateing adj matrix usin

In [27]:
final_df = pd.concat(all_filtered, ignore_index=True)



In [ ]:
# Save to CSV
final_df.to_csv('svg-enhanced-MB.csv', index=False)